# LeetCode 1266 — Minimum Time Visiting All Points bằng Python

**Link đề:** https://leetcode.com/problems/minimum-time-visiting-all-points/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Geometry

```python
class Solution:
    def minTimeToVisitAllPoints(self, points: list[list[int]]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu luật di chuyển 8 hướng (như **quân Vua** trong cờ vua) và vì sao đi chéo "tiết kiệm".
2. Viết lời giải **mô phỏng từng giây** (tham lam: còn lệch cả 2 trục thì đi chéo) để có trực giác.
3. Rút ra công thức **`max(|dx|, |dy|)`** — khoảng cách **Chebyshev** — và chứng minh nó là tối thiểu (cận dưới + cách đạt cận).
4. Nắm công cụ Python: `abs`, `max`, unpacking, `zip(points, points[1:])`, `itertools.pairwise`, `sum(...)` với generator.
5. Phân biệt với khoảng cách **Manhattan** và **Euclid**, test đối chiếu, lỗi hay gặp, biến thể và bài tập có đáp án.

## 1. Đề bài và ví dụ

Trên mặt phẳng có `n` điểm tọa độ nguyên `points[i] = [xi, yi]`. Trả về **thời gian nhỏ nhất (giây)** để đi qua tất cả các điểm **theo đúng thứ tự** trong mảng.

Mỗi giây, bạn được:
- đi dọc 1 đơn vị, **hoặc**
- đi ngang 1 đơn vị, **hoặc**
- đi chéo `sqrt(2)` đơn vị (tức là 1 đơn vị dọc **và** 1 đơn vị ngang cùng lúc).

Được phép đi ngang qua các điểm phía sau trong thứ tự, nhưng không tính là đã thăm.

| `points` | Output |
|---|---:|
| `[[1,1],[3,4],[-1,0]]` | `7` |
| `[[3,2],[-2,2]]` | `5` |

**Constraints:** `1 <= n <= 100`, `-1000 <= xi, yi <= 1000`.

Ví dụ 1 có một đường tối ưu: `[1,1] → [2,2] → [3,3] → [3,4] → [2,3] → [1,2] → [0,1] → [-1,0]` — 3 giây cho đoạn đầu, 4 giây cho đoạn sau.

### Quan sát đầu tiên: bài tách thành từng đoạn

Vì phải thăm theo thứ tự, tổng thời gian = **tổng thời gian từng đoạn** `points[i] → points[i+1]`.
Đoạn nào cũng nên đi nhanh nhất có thể, và đoạn sau không bị ảnh hưởng bởi cách đi đoạn trước (luôn bắt đầu đúng tại `points[i]`).

→ Bài toán thu nhỏ: **thời gian ít nhất đi từ điểm A tới điểm B** là bao nhiêu?

In [1]:
points = [[1, 1], [3, 4], [-1, 0]]
for i in range(len(points) - 1):
    print(f"đoạn {i}: {points[i]} -> {points[i + 1]}")

đoạn 0: [1, 1] -> [3, 4]
đoạn 1: [3, 4] -> [-1, 0]


## 2. Nền tảng Python cần biết

### 2.1. `abs` — độ lớn của độ lệch

`dx = x2 - x1` có thể âm (đi sang trái). Số bước cần đi theo trục x là `abs(dx)` — giá trị tuyệt đối.

In [2]:
x1, x2 = 3, -1
print(x2 - x1)          # -4  (đi sang trái 4)
print(abs(x2 - x1))     # 4
print(abs(-7), abs(7), abs(0))   # 7 7 0

-4
4
7 7 0


### 2.2. Unpacking và chỉ số

Mỗi điểm là list 2 phần tử. `x, y = [3, 4]` tách thành `x = 3`, `y = 4`. Trong vòng lặp: `for x, y in points`.

In [3]:
p = [3, 4]
x, y = p
print(x, y)
for x, y in [[1, 1], [3, 4]]:
    print("x =", x, "y =", y)

3 4
x = 1 y = 1
x = 3 y = 4


### 2.3. Duyệt **cặp liên tiếp**: `zip(points, points[1:])`

- `points[1:]` = list bỏ phần tử đầu.
- `zip(a, b)` ghép `a[0]` với `b[0]`, `a[1]` với `b[1]`, … và dừng ở list **ngắn hơn**.
- Nên `zip(points, points[1:])` cho đúng các cặp `(points[i], points[i+1])`.

Từ Python 3.10 có `itertools.pairwise(points)` làm đúng việc này.

In [4]:
points = [[1, 1], [3, 4], [-1, 0]]
print(points[1:])
for a, b_ in zip(points, points[1:]):
    print(a, "->", b_)

from itertools import pairwise
print(list(pairwise([10, 20, 30, 40])))   # [(10, 20), (20, 30), (30, 40)]

[[3, 4], [-1, 0]]
[1, 1] -> [3, 4]
[3, 4] -> [-1, 0]
[(10, 20), (20, 30), (30, 40)]


### 2.4. `max`, `min` và `sum` với generator

- `max(a, b)` / `min(a, b)` lấy số lớn / nhỏ.
- `sum(biểu_thức for ... in ...)` cộng dồn mà không cần tạo list trung gian.

In [5]:
print(max(2, 3), min(2, 3))                 # 3 2
print(sum(x * x for x in [1, 2, 3]))        # 14

3 2
14


## 3. Cách 1 — Mô phỏng từng giây (tham lam)

Trực giác: đi chéo xử lý **cả hai trục** trong 1 giây, còn đi thẳng chỉ xử lý **một trục**. Vậy:

- Khi còn lệch **cả x lẫn y**: đi chéo về phía đích.
- Khi chỉ còn lệch **một trục**: đi thẳng.

Để biết đi theo hướng nào, dùng **dấu** của độ lệch: `+1`, `-1` hoặc `0`.

In [6]:
def sign(v: int) -> int:
    # trả về 1 nếu v > 0, -1 nếu v < 0, 0 nếu v == 0
    return (v > 0) - (v < 0)

print(sign(5), sign(-3), sign(0))    # 1 -1 0
print(True - False, False - True)    # 1 -1  (bool là số 1/0)

1 -1 0
1 -1


In [7]:
def time_between_simulate(a, b):
    x, y = a
    tx, ty = b
    seconds = 0
    while (x, y) != (tx, ty):
        x += sign(tx - x)        # nếu đã đúng cột thì sign = 0 -> không đổi
        y += sign(ty - y)
        seconds += 1
    return seconds


def min_time_simulate(points):
    return sum(time_between_simulate(a, b) for a, b in zip(points, points[1:]))

print(time_between_simulate([1, 1], [3, 4]))     # 3
print(min_time_simulate([[1, 1], [3, 4], [-1, 0]]))   # 7
print(min_time_simulate([[3, 2], [-2, 2]]))           # 5

3
7
5


**Nhận xét:** mỗi vòng `while` ta cộng `sign(...)` cho **cả hai** trục: nếu cả hai khác 0 thì đó là bước chéo, nếu một trục đã khớp thì là bước thẳng. Rất gọn!

**Độ phức tạp Cách 1:** mỗi đoạn tốn số vòng bằng số giây của đoạn đó (tối đa 2000 với constraints) → `O(n · C)` với `C` là tọa độ lớn nhất. Chạy được, nhưng phụ thuộc **độ lớn tọa độ** — nếu tọa độ tới `10^9` thì quá chậm.

## 4. Nút thắt

Ta đang "đi bộ thật" từng giây chỉ để đếm. Có công thức nào cho **số vòng `while`** mà không cần chạy?

## 5. Insight — đếm số bước chéo và số bước thẳng

Gọi `dx = |x2 - x1|`, `dy = |y2 - y1|`.

- Số bước **chéo** = `min(dx, dy)` (đi chéo đến khi một trục khớp).
- Sau đó còn lệch `max(dx, dy) - min(dx, dy)` ở trục kia → bấy nhiêu bước **thẳng**.

```text
tổng = min(dx, dy) + (max(dx, dy) - min(dx, dy)) = max(dx, dy)
```

Ví dụ `[1,1] → [3,4]`: `dx = 2, dy = 3` → chéo 2, thẳng 1 → `3 = max(2, 3)`.

In [8]:
a, b_ = [1, 1], [3, 4]
dx, dy = abs(b_[0] - a[0]), abs(b_[1] - a[1])
diag = min(dx, dy)
straight = max(dx, dy) - diag
print(f"dx={dx}, dy={dy}, chéo={diag}, thẳng={straight}, tổng={diag + straight}, max={max(dx, dy)}")

dx=2, dy=3, chéo=2, thẳng=1, tổng=3, max=3


### Vì sao không thể nhanh hơn `max(dx, dy)`? (cận dưới)

Mỗi giây, tọa độ x thay đổi **nhiều nhất 1** (dù đi chéo hay thẳng). Muốn x thay đổi `dx` đơn vị cần **ít nhất `dx` giây**. Tương tự, cần ít nhất `dy` giây cho trục y. Nên cần ít nhất `max(dx, dy)` giây.

Mà chiến lược "chéo trước, thẳng sau" **đạt đúng** `max(dx, dy)` → đó là tối thiểu.

Khoảng cách `max(|dx|, |dy|)` có tên là **khoảng cách Chebyshev** (khoảng cách của quân Vua trên bàn cờ).

## 6. Cách 2 — Công thức Chebyshev (lời giải chính)

In [9]:
def min_time_chebyshev(points):
    total = 0
    for i in range(len(points) - 1):
        x1, y1 = points[i]
        x2, y2 = points[i + 1]
        total += max(abs(x2 - x1), abs(y2 - y1))
    return total

print(min_time_chebyshev([[1, 1], [3, 4], [-1, 0]]))   # 7
print(min_time_chebyshev([[3, 2], [-2, 2]]))           # 5
print(min_time_chebyshev([[0, 0]]))                    # 0 (chỉ 1 điểm)

7
5
0


### Phiên bản gọn với `zip` / `pairwise`

Cùng thuật toán, chỉ viết gọn hơn. Lưu ý: gọn không có nghĩa là nhanh hơn về Big-O.

In [10]:
from itertools import pairwise

def min_time_zip(points):
    return sum(max(abs(x2 - x1), abs(y2 - y1)) for (x1, y1), (x2, y2) in zip(points, points[1:]))

def min_time_pairwise(points):
    return sum(max(abs(b[0] - a[0]), abs(b[1] - a[1])) for a, b in pairwise(points))

print(min_time_zip([[1, 1], [3, 4], [-1, 0]]), min_time_pairwise([[1, 1], [3, 4], [-1, 0]]))

7 7


`for (x1, y1), (x2, y2) in zip(...)`: mỗi phần tử của `zip` là một cặp `(điểm_a, điểm_b)`; ta tách **hai tầng** cùng lúc — điểm a thành `x1, y1`, điểm b thành `x2, y2`.

## 7. Cách 3 — Biến đổi tọa độ (xoay 45°) — góc nhìn nâng cao

Mẹo toán học: đặt `u = x + y`, `v = x - y`. Khi đó

```text
max(|dx|, |dy|) = (|du| + |dv|) / 2
```

Tức là khoảng cách Chebyshev trên `(x, y)` = một nửa khoảng cách **Manhattan** trên `(u, v)`. Mẹo này rất hữu ích ở các bài khó (ví dụ tìm điểm xa nhất theo Chebyshev/Manhattan). Ở đây chỉ để mở rộng tầm nhìn, ta kiểm chứng bằng code.

In [11]:
def min_time_rotated(points):
    total2 = 0
    for (x1, y1), (x2, y2) in zip(points, points[1:]):
        du = (x2 + y2) - (x1 + y1)
        dv = (x2 - y2) - (x1 - y1)
        total2 += abs(du) + abs(dv)
    return total2 // 2     # du, dv cùng tính chẵn lẻ nên tổng mỗi đoạn luôn chẵn

print(min_time_rotated([[1, 1], [3, 4], [-1, 0]]))   # 7
# kiểm chứng đẳng thức trên nhiều cặp
import random
random.seed(0)
for _ in range(1000):
    dx, dy = random.randint(-50, 50), random.randint(-50, 50)
    assert max(abs(dx), abs(dy)) * 2 == abs(dx + dy) + abs(dx - dy)
print("đẳng thức đúng trên 1000 cặp")

7
đẳng thức đúng trên 1000 cặp


Vì sao đẳng thức đúng? Giả sử `|dx| >= |dy|`. Khi đó `dx + dy` và `dx - dy` cùng dấu với `dx`, nên `|dx + dy| + |dx - dy| = |2·dx| = 2·max`. Trường hợp ngược lại tương tự.

## 8. Trace từng bước

Ví dụ 1 `[[1,1],[3,4],[-1,0]]`:

| đoạn | từ → đến | `dx` | `dy` | chéo `min` | thẳng `max−min` | giây `max` | tổng dồn |
|---:|---|---:|---:|---:|---:|---:|---:|
| 0 | `[1,1] → [3,4]` | 2 | 3 | 2 | 1 | 3 | 3 |
| 1 | `[3,4] → [-1,0]` | 4 | 4 | 4 | 0 | 4 | 7 |

In [12]:
def trace_min_time(points):
    total = 0
    for i, ((x1, y1), (x2, y2)) in enumerate(zip(points, points[1:])):
        dx, dy = abs(x2 - x1), abs(y2 - y1)
        t = max(dx, dy)
        total += t
        print(f"đoạn {i}: {[x1, y1]} -> {[x2, y2]} | dx={dx}, dy={dy} | chéo={min(dx, dy)}, thẳng={t - min(dx, dy)} | +{t} -> tổng={total}")
    return total

trace_min_time([[1, 1], [3, 4], [-1, 0]])

đoạn 0: [1, 1] -> [3, 4] | dx=2, dy=3 | chéo=2, thẳng=1 | +3 -> tổng=3
đoạn 1: [3, 4] -> [-1, 0] | dx=4, dy=4 | chéo=4, thẳng=0 | +4 -> tổng=7


7

### Trace đường đi thật (để thấy chiến lược chéo trước)

In [13]:
def trace_path(a, b):
    x, y = a
    path = [(x, y)]
    while (x, y) != tuple(b):
        x += sign(b[0] - x)
        y += sign(b[1] - y)
        path.append((x, y))
    print(" -> ".join(map(str, path)), f"| {len(path) - 1} giây")

trace_path([1, 1], [3, 4])
trace_path([3, 4], [-1, 0])
trace_path([3, 2], [-2, 2])

(1, 1) -> (2, 2) -> (3, 3) -> (3, 4) | 3 giây
(3, 4) -> (2, 3) -> (1, 2) -> (0, 1) -> (-1, 0) | 4 giây
(3, 2) -> (2, 2) -> (1, 2) -> (0, 2) -> (-1, 2) -> (-2, 2) | 5 giây


`tuple(b)` đổi list `[3, 4]` thành tuple `(3, 4)` để so sánh với `(x, y)` — vì `(3, 4) != [3, 4]` trong Python (khác kiểu).
`map(str, path)` áp `str` cho từng phần tử, `" -> ".join(...)` nối chúng bằng chuỗi `" -> "`.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minTimeToVisitAllPoints(self, points: list[list[int]]) -> int:
        total = 0
        for i in range(1, len(points)):
            dx = abs(points[i][0] - points[i - 1][0])
            dy = abs(points[i][1] - points[i - 1][1])
            total += max(dx, dy)
        return total

print(Solution().minTimeToVisitAllPoints([[1, 1], [3, 4], [-1, 0]]))   # 7
print(Solution().minTimeToVisitAllPoints([[3, 2], [-2, 2]]))           # 5

7
5


## 10. Vì sao đúng?

1. **Tách đoạn:** mọi lộ trình hợp lệ phải đi qua `p0, p1, …, p(n-1)` theo thứ tự, nên thời gian ≥ tổng thời gian tối thiểu của từng đoạn. Và ta đạt được tổng đó bằng cách đi tối ưu từng đoạn nối tiếp → tổng các tối thiểu là đáp án.
2. **Mỗi đoạn:** cận dưới `max(dx, dy)` (mỗi giây mỗi trục đổi ≤ 1) + chiến lược đạt cận (chéo `min` bước, thẳng `max - min` bước).
3. **Invariant vòng lặp:** sau khi xử lý `i` đoạn đầu, `total` = thời gian tối thiểu để thăm `p0..pi` theo thứ tự.

## 11. Độ phức tạp và so sánh

Gọi `n = len(points)`, `C` = độ lớn tọa độ tối đa.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng giây | `O(n · C)` | `O(1)` | trực quan, phụ thuộc độ lớn tọa độ |
| 2. Chebyshev `max(dx, dy)` | `O(n)` | `O(1)` | lời giải chính |
| 2'. `zip`/`pairwise` | `O(n)` | `O(n)` cho `points[1:]` / `O(1)` với `pairwise` | gọn |
| 3. Xoay 45° | `O(n)` | `O(1)` | góc nhìn toán học |

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.minTimeToVisitAllPoints([[1, 1], [3, 4], [-1, 0]]), 7)
check("đề 2", sol.minTimeToVisitAllPoints([[3, 2], [-2, 2]]), 5)
check("1 điểm", sol.minTimeToVisitAllPoints([[5, -5]]), 0)
check("trùng điểm", sol.minTimeToVisitAllPoints([[2, 2], [2, 2], [2, 2]]), 0)
check("chéo thuần", sol.minTimeToVisitAllPoints([[0, 0], [5, 5]]), 5)
check("dọc thuần", sol.minTimeToVisitAllPoints([[0, 0], [0, -7]]), 7)
check("đi rồi quay lại", sol.minTimeToVisitAllPoints([[0, 0], [3, 1], [0, 0]]), 6)
check("biên tọa độ", sol.minTimeToVisitAllPoints([[-1000, -1000], [1000, 1000]]), 2000)

PASS đề 1: 7
PASS đề 2: 5
PASS 1 điểm: 0
PASS trùng điểm: 0
PASS chéo thuần: 5
PASS dọc thuần: 7
PASS đi rồi quay lại: 6
PASS biên tọa độ: 2000


In [16]:
import random
random.seed(1266)
methods = [min_time_simulate, min_time_chebyshev, min_time_zip, min_time_pairwise, min_time_rotated]
for _ in range(300):
    n = random.randint(1, 8)
    pts = [[random.randint(-30, 30), random.randint(-30, 30)] for _ in range(n)]
    expected = min_time_simulate(pts)
    for f in methods:
        assert f(pts) == expected, (f.__name__, pts)
    assert Solution().minTimeToVisitAllPoints(pts) == expected
print("300 bộ ngẫu nhiên: 5 cách + Solution đều khớp")

300 bộ ngẫu nhiên: 5 cách + Solution đều khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng khoảng cách Manhattan `dx + dy`

Đó là thời gian khi **không được đi chéo**. Ở bài này đi chéo cũng chỉ tốn 1 giây.

In [17]:
def wrong_manhattan(points):
    return sum(abs(b[0] - a[0]) + abs(b[1] - a[1]) for a, b in zip(points, points[1:]))

print("sai :", wrong_manhattan([[1, 1], [3, 4], [-1, 0]]))   # 13
print("đúng:", Solution().minTimeToVisitAllPoints([[1, 1], [3, 4], [-1, 0]]))

sai : 13
đúng: 7


### Lỗi 2: Dùng khoảng cách Euclid `sqrt(dx² + dy²)`

Đề đếm **giây**, không đo **độ dài**. Bước chéo dài `sqrt(2)` nhưng chỉ tốn 1 giây. Euclid còn trả số thực.

In [18]:
import math

def wrong_euclid(points):
    return sum(math.dist(a, b) for a, b in zip(points, points[1:]))

print("sai :", round(wrong_euclid([[1, 1], [3, 4], [-1, 0]]), 3))
print("đúng:", Solution().minTimeToVisitAllPoints([[1, 1], [3, 4], [-1, 0]]))

sai : 9.262
đúng: 7


`math.dist(a, b)` (Python 3.8+) tính khoảng cách Euclid giữa hai điểm.

### Lỗi 3: Quên `abs`

`max(x2 - x1, y2 - y1)` sai khi điểm sau nằm bên trái/phía dưới: độ lệch âm bị "thua" trong `max`.

In [19]:
def wrong_no_abs(points):
    return sum(max(b[0] - a[0], b[1] - a[1]) for a, b in zip(points, points[1:]))

print("sai :", wrong_no_abs([[3, 2], [-2, 2]]))     # max(-5, 0) = 0
print("đúng:", Solution().minTimeToVisitAllPoints([[3, 2], [-2, 2]]))

sai : 0
đúng: 5


### Lỗi 4: Lấy `max` của `|dx|` và `|dy|` **trên toàn bộ** thay vì từng đoạn

Ví dụ tính `max(tổng|dx|, tổng|dy|)`: sai vì mỗi đoạn có thể bị chi phối bởi trục khác nhau.

In [20]:
def wrong_global_max(points):
    sx = sum(abs(b[0] - a[0]) for a, b in zip(points, points[1:]))
    sy = sum(abs(b[1] - a[1]) for a, b in zip(points, points[1:]))
    return max(sx, sy)

pts = [[0, 0], [5, 0], [5, 5]]       # đoạn 1 chỉ đi ngang 5, đoạn 2 chỉ đi dọc 5
print("sai :", wrong_global_max(pts))     # 5
print("đúng:", Solution().minTimeToVisitAllPoints(pts))   # 10

sai : 5
đúng: 10


### Lỗi 5: Vòng lặp `range(len(points))` rồi đọc `points[i + 1]`

Ở `i` cuối cùng, `points[i + 1]` vượt chỉ số → `IndexError`. Dùng `range(len(points) - 1)` hoặc `range(1, len(points))` với `points[i - 1]`.

In [21]:
def wrong_index(points):
    total = 0
    for i in range(len(points)):           # SAI: thừa 1 vòng
        total += max(abs(points[i + 1][0] - points[i][0]), abs(points[i + 1][1] - points[i][1]))
    return total

try:
    wrong_index([[1, 1], [3, 4]])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:**
- Di chuyển 8 hướng, mỗi bước chi phí bằng nhau → **Chebyshev** `max(|dx|, |dy|)`.
- Chỉ 4 hướng (lên/xuống/trái/phải) → **Manhattan** `|dx| + |dy|`.
- Đi tự do theo đường thẳng, đo độ dài → **Euclid**.
- "Thăm theo thứ tự cho trước" → tổng các đoạn liên tiếp, duyệt **cặp kề nhau**.

**Biến thể / bài liên quan:**
- Số bước quân Vua trên bàn cờ (cùng công thức).
- LeetCode 1217 *Minimum Cost to Move Chips to The Same Position*: một bài "chi phí di chuyển" với quy tắc bước khác (bước 2 miễn phí).
- LeetCode 2849 *Determine if a Cell Is Reachable at a Given Time*: đúng Chebyshev + xử lý trường hợp đặc biệt `t = 1` khi đứng yên.
- LeetCode 1584 *Min Cost to Connect All Points* (Manhattan + cây khung nhỏ nhất) — khi **không** có thứ tự cho trước.

In [22]:
# Mini so sánh 3 loại khoảng cách cho cùng một cặp điểm
a, b_ = (0, 0), (3, 4)
dx, dy = abs(b_[0] - a[0]), abs(b_[1] - a[1])
print("Chebyshev:", max(dx, dy))        # 4
print("Manhattan:", dx + dy)            # 7
print("Euclid   :", math.dist(a, b_))   # 5.0

Chebyshev: 4
Manhattan: 7
Euclid   : 5.0


## 15. Bài tập tự luyện

> Gợi ý ở mục 16, đáp án ở mục 17. Cell bài tập **không** được gọi khi chạy notebook.

### Bài 1 — Tính tay
Thời gian tối thiểu cho `[[0,0],[2,-3],[2,-3],[-4,1]]` là bao nhiêu? Gán vào `ex1_answer`.

In [23]:
ex1_answer = None   # TODO

# assert ex1_answer == Solution().minTimeToVisitAllPoints([[0, 0], [2, -3], [2, -3], [-4, 1]])

### Bài 2 — Chỉ được đi 4 hướng
Viết `min_time_4dir(points)`: không được đi chéo.

In [24]:
def min_time_4dir(points: list[list[int]]) -> int:
    # TODO
    pass

# assert min_time_4dir([[1, 1], [3, 4], [-1, 0]]) == 13

### Bài 3 — Trả về số bước chéo và số bước thẳng
Viết `count_moves(points)` trả về tuple `(so_buoc_cheo, so_buoc_thang)` của lộ trình tối ưu "chéo trước".

In [25]:
def count_moves(points: list[list[int]]) -> tuple[int, int]:
    # TODO
    pass

# assert count_moves([[1, 1], [3, 4], [-1, 0]]) == (6, 1)

### Bài 4 — Đi một vòng khép kín
Sau khi thăm điểm cuối, phải **quay về điểm đầu**. Viết `min_time_closed_tour(points)`.

In [26]:
def min_time_closed_tour(points: list[list[int]]) -> int:
    # TODO
    pass

# assert min_time_closed_tour([[1, 1], [3, 4], [-1, 0]]) == 7 + 2
# assert min_time_closed_tour([[0, 0]]) == 0

### Bài 5 — Mini challenge: đến đúng lúc
Viết `reachable_at(sx, sy, fx, fy, t)`: mỗi giây **bắt buộc** di chuyển sang 1 trong 8 ô kề. Có thể đứng ở `(fx, fy)` **đúng** sau `t` giây không? (LeetCode 2849)

In [27]:
def reachable_at(sx: int, sy: int, fx: int, fy: int, t: int) -> bool:
    # TODO: so sánh t với khoảng cách Chebyshev, cẩn thận trường hợp điểm đầu == điểm cuối
    pass

# assert reachable_at(2, 4, 7, 7, 6) == True
# assert reachable_at(3, 1, 7, 3, 3) == False
# assert reachable_at(1, 1, 1, 1, 1) == False
# assert reachable_at(1, 1, 1, 1, 2) == True

## 16. Gợi ý 3 mức

**Bài 1:** (1) Tách 3 đoạn. (2) Đoạn giữa trùng điểm = 0 giây. (3) `max(2,3) + 0 + max(6,4)`.

**Bài 2:** (1) Không có bước chéo. (2) Mỗi trục đi riêng. (3) `abs(dx) + abs(dy)`.

**Bài 3:** (1) Mục 5. (2) Chéo = `min(dx, dy)`, thẳng = `max - min`. (3) Cộng dồn hai biến qua các đoạn.

**Bài 4:** (1) Thêm một đoạn cuối → đầu. (2) `points + [points[0]]`. (3) Dùng lại hàm Chebyshev.

**Bài 5:** (1) Ít nhất `d = max(|dx|, |dy|)` giây. (2) Thừa thời gian thì "đi vòng" được vì luôn có ô kề để đi ra rồi quay lại. (3) Nếu `d == 0`: cần `t != 1`; ngược lại `t >= d`.

## 17. Đáp án tham khảo

In [28]:
ans_ex1 = 3 + 0 + 6


def ans_min_time_4dir(points):
    return sum(abs(b[0] - a[0]) + abs(b[1] - a[1]) for a, b in zip(points, points[1:]))


def ans_count_moves(points):
    diag = straight = 0
    for (x1, y1), (x2, y2) in zip(points, points[1:]):
        dx, dy = abs(x2 - x1), abs(y2 - y1)
        diag += min(dx, dy)
        straight += max(dx, dy) - min(dx, dy)
    return diag, straight


def ans_min_time_closed_tour(points):
    loop = points + [points[0]]
    return sum(max(abs(b[0] - a[0]), abs(b[1] - a[1])) for a, b in zip(loop, loop[1:]))


def ans_reachable_at(sx, sy, fx, fy, t):
    d = max(abs(fx - sx), abs(fy - sy))
    if d == 0:
        return t != 1
    return t >= d


assert ans_ex1 == Solution().minTimeToVisitAllPoints([[0, 0], [2, -3], [2, -3], [-4, 1]]) == 9
assert ans_min_time_4dir([[1, 1], [3, 4], [-1, 0]]) == 13
assert ans_count_moves([[1, 1], [3, 4], [-1, 0]]) == (6, 1)
assert ans_min_time_closed_tour([[1, 1], [3, 4], [-1, 0]]) == 9
assert ans_min_time_closed_tour([[0, 0]]) == 0
assert ans_reachable_at(2, 4, 7, 7, 6) == True
assert ans_reachable_at(3, 1, 7, 3, 3) == False
assert ans_reachable_at(1, 1, 1, 1, 1) == False
assert ans_reachable_at(1, 1, 1, 1, 2) == True

# kiểm tra chéo: số bước chéo + thẳng = đáp án Chebyshev
import random
random.seed(9)
for _ in range(200):
    pts = [[random.randint(-20, 20), random.randint(-20, 20)] for _ in range(random.randint(1, 6))]
    d, s = ans_count_moves(pts)
    assert d + s == Solution().minTimeToVisitAllPoints(pts)
print("Đáp án bài tập: tất cả assert PASS")

Đáp án bài tập: tất cả assert PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minTimeToVisitAllPoints(self, points):
        total = 0
        for i in range(1, len(points)):
            dx = abs(points[i][0] - points[i - 1][0])
            dy = abs(points[i][1] - points[i - 1][1])
            total += max(dx, dy)
        return total
```

**Câu thần chú:** *"8 hướng → max; 4 hướng → cộng; thứ tự cố định → cộng từng đoạn."*

**Checklist:**
- [ ] Dùng `abs` cho cả `dx` và `dy`.
- [ ] Dùng `max(dx, dy)` (không phải `dx + dy`, không phải Euclid).
- [ ] Duyệt **cặp liên tiếp**, không vượt chỉ số.
- [ ] `n = 1` trả `0`.
- [ ] Đã test ví dụ đề + điểm trùng + đi ngược chiều (độ lệch âm).